# miniSAM3D on Colab: label with SAM 3D Body, train the student

**Runtime:** GPU. Labeling (the 840M-parameter teacher) wants an **A100 or L4** (Colab Pro); a free T4 works but is slow.
Training the 2M-parameter student is fine on a T4.

The code is cloned from GitHub. Everything durable (labels, checkpoints, your recordings) lives in Google Drive under `MyDrive/minisam3d*`,
and every long step resumes where it stopped, so a disconnect only costs the work since the last shard or epoch.

Before you start:
1. Request access to https://huggingface.co/facebook/sam-3d-body-dinov3 and create a read token.
2. Optional, for your own footage: put fisheye recordings in `MyDrive/minisam3d_rec/` and the calibration
   from `scripts/calibrate_fisheye.py` at `MyDrive/minisam3d_rec/calib_left.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive'
REC = f'{D}/minisam3d_rec'
LABELS, RUNS = f'{D}/minisam3d_labels', f'{D}/minisam3d_runs'
DATA = '/content/data'  # local disk: fast, wiped on disconnect
for p in (LABELS, RUNS, DATA): os.makedirs(p, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = 'https://github.com/SujithChristopher/minisam3d'
!test -d /content/minisam3d/.git && git -C /content/minisam3d pull -q || git clone -q {REPO} /content/minisam3d
%cd /content/minisam3d
!pip -q install opencv-python-headless onnx onnxruntime qai-hub

## 1. Install the teacher (once per runtime, ~10 min, mostly detectron2)

In [ ]:
%cd /content
!test -d sam-3d-body || git clone -q https://github.com/facebookresearch/sam-3d-body
!pip -q install pytorch-lightning pyrender yacs scikit-image einops timm dill pandas rich hydra-core pyrootutils \
    webdataset chump networkx==3.2.1 roma joblib appdirs jsonlines xtcocotools loguru optree fvcore pycocotools huggingface_hub
!pip -q install 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps
!pip -q install git+https://github.com/microsoft/MoGe.git
%cd /content/minisam3d

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste the read token for the gated facebook/sam-3d-body-dinov3 repo

## 2. Data
### 2a. COCO people (general pose variety)
Downloads train2017 (~18 GB) to local disk and links only images that contain a reasonably large person.
`N_COCO` caps how many go to the teacher; start small to measure teacher speed on your GPU.

In [ ]:
N_COCO = 30000
MIN_H = 200  # person box height in px
if not os.path.exists(f'{DATA}/train2017'):
    !wget -q -P {DATA} http://images.cocodataset.org/zips/train2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !cd {DATA} && unzip -q train2017.zip && unzip -q annotations_trainval2017.zip && rm train2017.zip annotations_trainval2017.zip
import json, random
ann = json.load(open(f'{DATA}/annotations/person_keypoints_train2017.json'))
ok = sorted({a['image_id'] for a in ann['annotations'] if a['bbox'][3] >= MIN_H and not a['iscrowd']})
random.Random(0).shuffle(ok)
names = {im['id']: im['file_name'] for im in ann['images']}
os.makedirs(f'{DATA}/coco_person', exist_ok=True)
for i in ok[:N_COCO]:
    dst = f'{DATA}/coco_person/{names[i]}'
    if not os.path.lexists(dst): os.symlink(f'{DATA}/train2017/{names[i]}', dst)
print(len(ok), 'candidate images,', len(os.listdir(f'{DATA}/coco_person')), 'linked')

### 2b. Your OV9281 footage (most valuable per frame)
Renders each fisheye frame into three 90-degree pinhole views (left / centre / right) with known intrinsics,
so the teacher sees normal perspective images. The rendered views are cached to Drive as a tar.

In [ ]:
VIEWS = f'{DATA}/own_views'
tar = f'{REC}/own_views.tar'
if os.path.exists(tar) and not os.path.exists(VIEWS):
    !tar -xf {tar} -C {DATA}
elif not os.path.exists(VIEWS):
    !python scripts/fisheye_to_views.py --calib {REC}/calib_left.json --videos '{REC}/*.mp4' '{REC}/*.avi' '{REC}/*.mkv' --out {VIEWS} --stride 5
    !tar -cf {tar} -C {DATA} own_views
!ls {VIEWS}/images | head

## 3. Label with SAM 3D Body (resumable: just re-run the cell after a disconnect)
Labels go to Drive in shards of 2000 people.

In [ ]:
TEACHER = '/content/sam-3d-body'
# your footage first: it is smaller and matters most
if os.path.exists(f'{DATA}/own_views'):
    !PYTHONPATH={TEACHER} python scripts/label_with_teacher.py --inputs {DATA}/own_views/images --out {LABELS}/own

In [ ]:
!PYTHONPATH={TEACHER} python scripts/label_with_teacher.py --inputs {DATA}/coco_person --out {LABELS}/coco

## 4. Train the student (resumable)
`--gray` makes a 1-channel model for the mono OV9281. `--repeat` oversamples your own footage.

In [ ]:
roots = [r for r in (f'{LABELS}/coco', f'{LABELS}/own') if os.path.exists(f'{r}/labels')]
rep = '1 4' if len(roots) == 2 else '1'
!python scripts/train.py --data {' '.join(roots)} --repeat {rep} --image-dirs {DATA} {DATA}/own_views \
    --gray --pretrained --epochs 60 --bs 128 --workers 4 --out {RUNS}/v1 --resume {RUNS}/v1/last.pt

## 5. Export and profile on a real QCS6490 (Qualcomm AI Hub)
Needs a free AI Hub API token from https://app.aihub.qualcomm.com (Settings → API token).

In [ ]:
AIHUB_TOKEN = ''  # paste here
!qai-hub configure --api_token {AIHUB_TOKEN} > /dev/null
!python scripts/export_onnx.py --ckpt {RUNS}/v1/best.pt --out {RUNS}/v1/student.onnx
!python scripts/aihub_compile.py --onnx {RUNS}/v1/student.onnx --gray --calib {roots[0]} --image-dirs {DATA} {DATA}/own_views \
    --runtime tflite --outdir {RUNS}/v1/aihub